In [6]:
from pathlib import Path
ROOT = Path("../..").resolve()

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(ROOT / 'src/data/raw/bs140513_032310.csv')

In [7]:
print((df.groupby('age')['fraud'].mean()*100).reset_index().rename(columns={'age':'Age','fraud' : 'Fraud Percent'}).sort_values(by='Fraud Percent'))

   Age  Fraud Percent
7  'U'       0.594228
6  '6'       0.974826
5  '5'       1.095112
1  '1'       1.185254
3  '3'       1.192815
2  '2'       1.251401
4  '4'       1.293281
0  '0'       1.957586


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split




# 1. Clean 'age' — xử lý whitespace
df['age'] = df['age'].astype(str).str.strip().str.strip("'")

# 2. Clean 'gender' tương tự
df['gender'] = df['gender'].astype(str).str.strip().str.strip("'")

# 3. Drop zero-variance features
df = df.drop(columns=['zipcodeOri', 'zipMerchant'])

# 4. Temporal split — theo thứ tự step, không random
df = df.sort_values('step').reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

# 5. Encode categoricals — fit trên train, transform cả hai
cat_cols = ['age', 'gender', 'category', 'merchant', 'customer']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    # Handle unseen labels trong test
    test_df[col] = test_df[col].astype(str).map(
        lambda x, le=le: le.transform([x])[0]
        if x in le.classes_ else -1
    )
    encoders[col] = le

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Fraud rate train: {train_df['fraud'].mean():.4f}")
print(f"Fraud rate test:  {test_df['fraud'].mean():.4f}")

Train: 475714 | Test: 118929
Fraud rate train: 0.0124
Fraud rate test:  0.0108
